In [ ]:
!pip install pandas
!pip install sdv==1.29.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os

os.chdir("drive")
os.chdir("MyDrive")
os.chdir("Colab Notebooks")
os.chdir("Dissertation")

import glob
files = sorted(glob.glob("electra_split/*"))
df0 = pd.read_csv(files[0])
dfs = [df0]
c = 0
for f in files[1:]:
  c = c+1
  df = pd.read_csv(f, header=None)
  df.columns = df0.columns # apply the first file's headers
  dfs.append(df)
  if c == 250:
    break

df = pd.concat(dfs, ignore_index=True)

df = df.groupby(df.sip)
df = df.get_group("10.70.38.53")
df = df.drop('dmac', axis=1)
df = df.drop('smac', axis=1)
df = df.drop('sip', axis=1)
df = df.drop('dip', axis=1)
df = df.drop('label', axis = 1)
df = df.drop('Time', axis=1)
df = df.drop('request', axis=1)
df = df.drop('error', axis=1)
df = df.drop('fc', axis=1)
print(df)

In [ ]:
from sdv.metadata import Metadata

metadata = Metadata.detect_from_dataframe(
    data=df,
    table_name='electra')

metadata.update_column(
    column_name='address',
    sdtype='categorical'
)

from sdv.single_table import CTGANSynthesizer

synthesizer = CTGANSynthesizer(
    metadata,
    enforce_min_max_values=False,
    enforce_rounding=True,
    verbose=True,
    epochs=46,
    batch_size=1500,
    generator_dim=(128, 128),
    discriminator_dim=(128, 128)
)
synthesizer.fit(df)

synthetic_data = synthesizer.sample(num_rows=10)


In [ ]:
synthesizer.save(
    filepath='synthesizer.pkl'
)